# Gradient and neuron interference

Read [the definitions](../docs/measurements.md) before interpreting these plots.
The saved arrays support three distinct views: conflicting minibatch gradients,
harm from the actual optimizer update, and cancellation of neuron prediction
changes. This notebook analyzes existing snapshots and never reruns training.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from neural_competition.analysis import load_checkpoint, load_metrics, neuron_prediction_changes
from neural_competition.diagnostics import alignment_arrays

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/neural_competition").exists())
OUT = ROOT / "runs/analysis"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
pd.set_option("display.float_format", lambda x: f"{x:.3e}")
metrics = load_metrics(ROOT)
core = metrics[metrics.collection == "teacher8"]
widths = sorted(core.width.unique())
colors = {w: plt.get_cmap("tab10")(i) for i, w in enumerate(widths)}
data = torch.load(ROOT / "runs/teacher8/data.pt", weights_only=True)


SEED = 0
selected = core[core.seed == SEED]


## Trajectories at equal time and comparable loss

Negative gradient dots predict first-order cross-batch harm. Finite-step harm is
measured separately. Residual magnitude and stage of learning are confounders;
inspect both update count and pre-update training MSE. Fractions near numerical
precision may reflect rounding and should not be read as strong competition.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for width, a in selected.groupby("width"):
    for row, metric in enumerate(["pair_cosine_negative_fraction", "probe_loss_increase_fraction"]):
        axes[row, 0].plot(a.updates, a[metric], label=str(width), color=colors[width])
        axes[row, 1].plot(a.train_mse_before, a[metric], ".-", color=colors[width])
        axes[row, 0].set(xscale="log", xlabel="Updates", ylabel=metric.replace("_", " "))
        axes[row, 1].set(xscale="log", xlabel="Pre-update training MSE", ylabel=metric.replace("_", " "))
for ax in axes.flat:
    ax.grid(alpha=.2)
for ax in axes[:, 1]:
    ax.invert_xaxis()
axes[0, 0].legend(title="Width", ncol=2)
fig.tight_layout()
fig.savefig(OUT / "interference_trajectories.png", bbox_inches="tight")
plt.show()


## Approximate loss matching and full pair distributions

Choose the nearest available checkpoint to training MSE 0.01. This is coarse
matching across 13 saved checkpoints: the displayed actual losses expose the
remaining mismatch. The 2,016 pair values per histogram are dependent, not 2,016
independent initializations. Raw dots depend on magnitude; cosines show orientation.


In [ ]:
TARGET_MSE = .01
snapshots, matches = {}, []
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for width, a in selected.groupby("width"):
    distance = np.abs(np.log(a.train_mse_before / TARGET_MSE))
    row = a.loc[distance.idxmin()]
    s = load_checkpoint(row.run_path, int(row.step))
    snapshots[width] = s
    dots, cosines = alignment_arrays(s["probe_gradients"])
    matches.append(dict(width=width, step=int(row.step), train_mse=row.train_mse_before,
                        target_ratio=row.train_mse_before/TARGET_MSE,
                        negative_fraction=float((dots < 0).double().mean())))
    axes[0].hist(dots.numpy(), bins=40, density=True, histtype="step", label=str(width), color=colors[width])
    axes[1].hist(cosines[torch.isfinite(cosines)].numpy(), bins=np.linspace(-1, 1, 41), density=True,
                 histtype="step", label=str(width), color=colors[width])
matches = pd.DataFrame(matches)
display(matches)
matches.to_csv(OUT / "loss_matching.csv", index=False)
for ax, xlabel in zip(axes, ["Gradient dot", "Gradient cosine"]):
    ax.set(xlabel=xlabel, ylabel="Density")
    ax.axvline(0, color="black", lw=1)
axes[1].legend(title="Width")
fig.tight_layout()
fig.savefig(OUT / "gradient_pair_distributions.png", bbox_inches="tight")
plt.show()


## Which neurons carry conflicting demands?

For each width, inspect the pair of probe batches with the most negative cosine.
Each bar is the block contribution q_r. Their sum is exactly the global gradient
dot, so negative contributions can be hidden by positive contributions elsewhere.
The worst pair differs between widths; these panels are illustrative attributions.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
for ax, width in zip(axes, [8, 10, 16]):
    s = snapshots[width]
    g = s["probe_gradients"].double()
    dots, cosines = alignment_arrays(g)
    worst = int(torch.nan_to_num(cosines, nan=2).argmin())
    indices = torch.triu_indices(len(g), len(g), offset=1)
    i, j = indices[:, worst].tolist()
    scores = (g[i] * g[j]).sum(dim=1)
    torch.testing.assert_close(scores.sum(), dots[worst])
    ax.bar(range(width), scores.numpy(), color=["C3" if v < 0 else "C0" for v in scores])
    ax.axhline(0, color="black", lw=1)
    ax.set(xlabel="Neuron", ylabel="Contribution to gradient dot",
           title=f"Width {width}; batches {i}, {j}")
fig.tight_layout()
fig.savefig(OUT / "neuron_demand_contributions.png", bbox_inches="tight")
plt.show()


## Predicted versus observed harm from one actual update

For delta = theta_after - theta_before, g_C dot delta predicts a change of each
probe batch's half-MSE. Positive means harm. The remainder includes smooth
higher-order terms and possible activation crossings; it is not purely quadratic.


In [ ]:
s = snapshots[10]
predicted = s["probe_gradients"].double().flatten(1) @ s["actual_step"].double().flatten()
observed = (s["probe_losses_after"] - s["probe_losses_before"]).double()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(predicted, observed, c=np.where(observed.numpy() > 0, "C3", "C0"), s=28)
lo = min(predicted.min().item(), observed.min().item())
hi = max(predicted.max().item(), observed.max().item())
axes[0].plot([lo, hi], [lo, hi], color="black", ls="--")
axes[0].axhline(0, color="gray", lw=.7)
axes[0].axvline(0, color="gray", lw=.7)
axes[0].set(xlabel="Predicted half-MSE change", ylabel="Observed half-MSE change", title=f"Width 10; step {s['step']}")
axes[1].hist((observed-predicted).numpy(), bins=25)
axes[1].set(xlabel="Observed minus linear prediction", ylabel="Probe batches")
fig.tight_layout()
fig.savefig(OUT / "finite_step_harm.png", bbox_inches="tight")
plt.show()
display(pd.DataFrame([dict(predicted_harm_fraction=float((predicted>0).double().mean()),
                           actual_harm_fraction=float((observed>0).double().mean()),
                           mean_remainder=float((observed-predicted).mean()))]))


## Consecutive updates and curvature

The consecutive actual-update cosine measures direction changes between adjacent
training steps. Curvature is the mean of four batch Rayleigh quotients for the
half-MSE. The line 2/lr is a plain-SGD reference, not a measurement of the full
Hessian's top eigenvalue or evidence of an edge-of-stability regime.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for width, a in selected.groupby("width"):
    axes[0].plot(a.updates, a.step_previous_cosine, label=str(width), color=colors[width])
    axes[1].plot(a.updates, a.batch_sharpness, color=colors[width])
axes[0].set(ylabel="Consecutive actual-update cosine")
axes[0].axhline(0, color="gray", lw=.7)
axes[1].set(ylabel="Mean batch sharpness of half-MSE")
axes[1].axhline(2, color="black", ls="--", label="2/lr; lr=1")
for ax in axes:
    ax.set(xscale="log", xlabel="Updates")
    ax.grid(alpha=.2)
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / "update_directions_and_curvature.png", bbox_inches="tight")
plt.show()


## Do neuron prediction changes cancel?

This is a different measurement from minibatch gradient conflict. We compute each
neuron's finite prediction change and linearization on the common probe inputs.
The cancellation ratio is squared net movement divided by the sum of squared
individual movements. Below one means cancellation; that can be useful compensation.

For neuron/example pairs with fixed activation gates, the remainder is exactly
the outgoing/incoming bilinear term. Crossings are marked separately. Float64
offline evaluation reduces subtraction error without changing trained parameters.


In [ ]:
x = data["x_train"][data["probe_indices"]].reshape(-1, 20)
prediction_rows = []
for width in [8, 10, 16]:
    s = snapshots[width]
    p = neuron_prediction_changes(s, x)
    ratios = {}
    for name in ["linear", "finite"]:
        v = p[name]
        denominator = v.square().mean(dim=0).sum()
        ratios[name] = float(v.sum(dim=1).square().mean()/denominator) if denominator > 0 else np.nan
    stable = ~p["gate_changed"].any(dim=1)
    remainder = (p["finite"] - p["linear"]).sum(dim=1)
    expected = p["quadratic_same_region"].sum(dim=1)
    if stable.any():
        torch.testing.assert_close(remainder[stable], expected[stable], atol=1e-12, rtol=1e-5)
    prediction_rows.append(dict(width=width, step=s["step"], linear_cancellation_ratio=ratios["linear"],
                                finite_cancellation_ratio=ratios["finite"],
                                gate_crossing_fraction=float(p["gate_changed"].double().mean()),
                                stable_examples=int(stable.sum())))
    if width == 10:
        v = p["linear"]
        gram = v.T @ v / len(v)
        norm = gram.diag().clamp_min(0).sqrt()
        denom = norm[:, None]*norm[None, :]
        neuron_cosine = torch.where(denom>0, gram/denom, torch.nan)
prediction_rows = pd.DataFrame(prediction_rows)
display(prediction_rows)
prediction_rows.to_csv(OUT / "neuron_prediction_changes.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, marker in [("linear", "o-"), ("finite", "s--")]:
    axes[0].plot(prediction_rows.width, prediction_rows[f"{name}_cancellation_ratio"], marker, label=name)
axes[0].axhline(1, color="black", lw=1, ls=":")
axes[0].set(xlabel="Width", ylabel="Prediction-movement ratio", xticks=[8, 10, 16])
axes[0].legend()
im = axes[1].imshow(neuron_cosine.numpy(), vmin=-1, vmax=1, cmap="coolwarm")
axes[1].set(xlabel="Neuron", ylabel="Neuron", title="Width 10: linearized movement cosine")
fig.colorbar(im, ax=axes[1], shrink=.85)
fig.tight_layout()
fig.savefig(OUT / "neuron_prediction_cancellation.png", bbox_inches="tight")
plt.show()


## What these diagnostics establish

The stored gradients attribute conflicting batch demands to neurons, and the
before/after snapshots measure actual finite-step harm and prediction compensation.
These are descriptive mechanisms to investigate. They do not establish that
interference causes failed optimization, that extra width always reduces conflict,
or that the deliberately selected seed subset gives an unbiased success rate.
For that rate, use the broader history in the training notebook.
